# Lecture 8 · Notebook 3: Generation and Chat Templates

Natural Language Processing (2026), SeoulTech.

We generate text with `generate`, compare a base model with its instruction-tuned version (Lecture 7),
look at the chat template, and build SFT labels with −100.

*Runtime: a GPU (Colab T4) is faster, but CPU works.*

In [ ]:
# Run this cell first. Colab already has PyTorch; this installs the tested library versions.
%pip install -q "transformers>=5.20,<6" "datasets>=5.1,<6" "accelerate>=1.10"

import torch, transformers, datasets
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("transformers", transformers.__version__, "| datasets", datasets.__version__, "| torch", torch.__version__)
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

## 1. Load a base model and an instruct model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# T4 has no fast bfloat16: use float16 on older GPUs, bfloat16 on newer ones, float32 on CPU
if device == "cuda":
    dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
else:
    dtype = torch.float32

base_name, inst_name = "HuggingFaceTB/SmolLM2-360M", "HuggingFaceTB/SmolLM2-360M-Instruct"
base_tok = AutoTokenizer.from_pretrained(base_name)
base = AutoModelForCausalLM.from_pretrained(base_name, dtype=dtype).to(device)
tok = AutoTokenizer.from_pretrained(inst_name)
inst = AutoModelForCausalLM.from_pretrained(inst_name, dtype=dtype).to(device)

## 2. `generate`

`generate` repeats the loop of Homework 2: run the model, pick the next token from the last position's
distribution, append it, and continue. The output contains the prompt **and** the new tokens.

In [ ]:
prompt = "The three most important ideas in natural language processing are"
inputs = base_tok(prompt, return_tensors="pt").to(device)
out = base.generate(**inputs, max_new_tokens=40, do_sample=False)     # greedy decoding
print(out.shape, inputs["input_ids"].shape)
new_tokens = out[0, inputs["input_ids"].shape[1]:]                     # drop the prompt
print(base_tok.decode(new_tokens, skip_special_tokens=True))

### Decoding options (Lecture 6)

| Argument | Meaning |
|---|---|
| `do_sample=False` | greedy (or beam search with `num_beams > 1`) |
| `do_sample=True, temperature=T` | sample from softmax(logits / T) |
| `top_k=k` | keep the k most probable tokens |
| `top_p=p` | keep the smallest set with total probability ≥ p |
| `max_new_tokens` | number of tokens to add (not counting the prompt) |
| `repetition_penalty` | lower the score of tokens that already appeared |

In [ ]:
torch.manual_seed(0)
for T in [0.3, 1.0, 1.5]:
    print(f"--- temperature {T}")
    outs = base.generate(**inputs, max_new_tokens=25, do_sample=True, temperature=T,
                         top_p=0.95, num_return_sequences=3)
    for o in outs:
        print(" ", base_tok.decode(o[inputs["input_ids"].shape[1]:], skip_special_tokens=True).replace("\n", " "))

## 3. Base model vs. instruct model

The example from Lecture 7, slide 3. The base model only continues the document;
the instruct model was post-trained (SFT + DPO) to answer.

In [ ]:
request = "Explain the moon landing to a 6 year old in a few sentences."

ids = base_tok(request, return_tensors="pt").to(device)
out = base.generate(**ids, max_new_tokens=60, do_sample=False)
print("BASE:\n", base_tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True))

messages = [{"role": "user", "content": request}]
ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(device)
out = inst.generate(**ids, max_new_tokens=100, do_sample=False)
print("\nINSTRUCT:\n", tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True))

## 4. What the chat template produces

`apply_chat_template(..., tokenize=False)` returns the string that the model actually sees.
The template is stored with the tokenizer (`tok.chat_template`) and differs between model families.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is BPE?"},
]
print(tok.apply_chat_template(messages, tokenize=False))
print("=" * 40)
print(tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

In [ ]:
print(tok.chat_template)                 # a Jinja template
print(tok.convert_ids_to_tokens([0, 1, 2]))
print("eos:", tok.eos_token, "pad:", tok.pad_token)

**Without a system message** this template inserts its own default system message. Try it:

In [ ]:
print(tok.apply_chat_template([{"role": "user", "content": "Hi"}], tokenize=False, add_generation_prompt=True))

### Multi-turn: append the reply, then the next question

In [ ]:
questions = ["Name one sorting algorithm.", "What is its time complexity?", "Give a one-line Python example."]
messages = []
for q in questions:
    messages.append({"role": "user", "content": q})
    ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(device)
    out = inst.generate(**ids, max_new_tokens=60, do_sample=False)
    reply = tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True)
    print("USER:", q, "\nASSISTANT:", reply, "\n")
    messages.append({"role": "assistant", "content": reply})   # the next turn sees the whole history

## 5. Batch generation: pad on the left

A decoder-only model writes after the **last** position. With right padding, the new tokens of
the shorter prompt would come after its pad tokens. So we pad on the left for generation.

In [ ]:
tok.padding_side = "left"
prompts = [[{"role": "user", "content": "Say hello in French."}],
           [{"role": "user", "content": "List three prime numbers greater than 100, separated by commas."}]]
batch = tok.apply_chat_template(prompts, add_generation_prompt=True, padding=True, return_tensors="pt").to(device)
print(batch["attention_mask"])
out = inst.generate(**batch, max_new_tokens=40, do_sample=False)
for row in out:
    print(tok.decode(row[batch["input_ids"].shape[1]:], skip_special_tokens=True))
tok.padding_side = "right"

## 6. From a chat template to SFT labels

Lecture 7, slide 12: the SFT loss is the next-token loss on the **response tokens only**.
We tokenize the prompt part and the whole conversation, then set the prompt positions to −100.

In [ ]:
example = [
    {"role": "user", "content": "What is the capital of South Korea?"},
    {"role": "assistant", "content": "The capital of South Korea is Seoul."},
]
prompt_ids = tok.apply_chat_template(example[:1], add_generation_prompt=True)["input_ids"]
full_ids = tok.apply_chat_template(example)["input_ids"]
assert full_ids[:len(prompt_ids)] == prompt_ids          # the prompt is a prefix of the full text

input_ids = torch.tensor([full_ids], device=device)
labels = input_ids.clone()
labels[0, :len(prompt_ids)] = -100                        # no loss on system, user, and "assistant" header

for t, l in zip(tok.convert_ids_to_tokens(full_ids), labels[0].tolist()):
    print(f"{t!r:>22}  {'loss' if l != -100 else '-'}")

In [ ]:
with torch.no_grad():
    loss_all = inst(input_ids=input_ids, labels=input_ids).loss
    loss_resp = inst(input_ids=input_ids, labels=labels).loss
print(f"loss on every token: {loss_all.item():.3f}   loss on the response only: {loss_resp.item():.3f}")

SFT minimizes `loss_resp` over many such examples. Libraries such as TRL (`SFTTrainer`) build these
labels for you; now you know what they contain.

## Try it

1. Generate with `temperature=0.01` and `do_sample=True`. Is the output the same as greedy decoding?
2. Ask the base model the moon-landing request again, but write the prompt as the start of a document
   that a 6-year-old would read (Lecture 7, slide 4). Does it help?
3. In Section 6, add a second turn to `example`. Which positions should get a loss?